# Preprocess the data & save

Input schema:  
`[symbol, date, open, high, low, close, volume_usd]`


In [3]:
import pandas as pd
import numpy as np

In [4]:
df = pd.read_csv('../data/coins.csv')
df.head()

,unix,date,symbol,open,high,low,close,volume_usd
0,1523937600,2018-04-17 04:00:00,ADA/USD,0.25551,0.28800,0.25551,0.26664,2165077
1,1523941200,2018-04-17 05:00:00,ADA/USD,0.26660,0.27798,0.26010,0.26200,2235633
2,1523944800,2018-04-17 06:00:00,ADA/USD,0.26221,0.26396,0.24800,0.25664,2153964
3,1523948400,2018-04-17 07:00:00,ADA/USD,0.25662,0.26300,0.25489,0.25698,1215621
4,1523952000,2018-04-17 08:00:00,ADA/USD,0.25636,0.25998,0.25229,0.25631,896096


In [5]:
df.shape

(844335, 8)

In [6]:
df['date'] = pd.to_datetime(df['date'])

In [7]:
print(f'Min: {df["date"].min()}\nMax: {df["date"].max()}')

Min: 2017-08-17 04:00:00
Max: 2023-10-19 23:00:00


In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 844335 entries, 0 to 844334
Data columns (total 8 columns):
 #   Column      Non-Null Count   Dtype         
---  ------      --------------   -----         
 0   unix        844335 non-null  int64         
 1   date        844335 non-null  datetime64[ns]
 2   symbol      844335 non-null  object        
 3   open        844335 non-null  float64       
 4   high        844335 non-null  float64       
 5   low         844335 non-null  float64       
 6   close       844335 non-null  float64       
 7   volume_usd  844335 non-null  int64         
dtypes: datetime64[ns](1), float64(4), int64(2), object(1)
memory usage: 51.5+ MB


## token extraction (BTC, ETH)


In [9]:
# Feature extraction; we do not intend to trade these
# symbols because the market is too efficient.
btc = df[df['symbol'] == 'BTC/USD']
eth = df[df['symbol'] == 'ETH/USD']

# Keep only close price for calculations
to_drop = ['open', 'high', 'low', 'volume_usd', 'symbol']
btc = btc[['date', 'close']].rename(columns={'close': 'btc_close'})
eth = eth[['date', 'close']].rename(columns={'close': 'eth_close'})

btc.info()

<class 'pandas.core.frame.DataFrame'>
Index: 54116 entries, 239332 to 293447
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   date       54116 non-null  datetime64[ns]
 1   btc_close  54116 non-null  float64       
dtypes: datetime64[ns](1), float64(1)
memory usage: 1.2 MB


In [10]:
print(len(df['symbol'].unique()))
df = df[~df['symbol'].isin({'ETH/USD', 'BTC/USD'})]
print(len(df['symbol'].unique()))

21
19


## next_close


In [11]:
# Create the target feature
df['next_close'] = df.groupby('symbol')['close'].shift(-1)
df.head()

,unix,date,symbol,open,high,low,close,volume_usd,next_close
0,1523937600,2018-04-17 04:00:00,ADA/USD,0.25551,0.28800,0.25551,0.26664,2165077,0.26200
1,1523941200,2018-04-17 05:00:00,ADA/USD,0.26660,0.27798,0.26010,0.26200,2235633,0.25664
2,1523944800,2018-04-17 06:00:00,ADA/USD,0.26221,0.26396,0.24800,0.25664,2153964,0.25698
3,1523948400,2018-04-17 07:00:00,ADA/USD,0.25662,0.26300,0.25489,0.25698,1215621,0.25631
4,1523952000,2018-04-17 08:00:00,ADA/USD,0.25636,0.25998,0.25229,0.25631,896096,0.25536


## OHLCV_log (log returns)


In [12]:
log_returns = lambda s: np.log(s / s.shift(1))  # noqa: E731
df['open_log'] = df.groupby('symbol')['open'].transform(log_returns)
df['high_log'] = df.groupby('symbol')['high'].transform(log_returns)
df['low_log'] = df.groupby('symbol')['low'].transform(log_returns)
df['close_log'] = df.groupby('symbol')['close'].transform(log_returns)

# The previous next close is the current close price so we can reverse by:
# predicted_next_close = current_close * exp(predicted_next_close_log)
df['next_close_log'] = df.groupby('symbol')['next_close'].transform(log_returns)

# Volume log uses a different formula simply to shrink the range
df['volume_log'] = df.groupby('symbol')['volume_usd'].transform(lambda x: np.log(1 + x))

In [13]:
df.head()

,unix,date,symbol,open,high,low,close,volume_usd,next_close,open_log,high_log,low_log,close_log,next_close_log,volume_log
0,1523937600,2018-04-17 04:00:00,ADA/USD,0.25551,0.28800,0.25551,0.26664,2165077,0.26200,NaN,NaN,NaN,NaN,NaN,14.587967
1,1523941200,2018-04-17 05:00:00,ADA/USD,0.26660,0.27798,0.26010,0.26200,2235633,0.25664,0.042488,-0.035411,0.017805,-0.017555,-0.020670,14.620035
2,1523944800,2018-04-17 06:00:00,ADA/USD,0.26221,0.26396,0.24800,0.25664,2153964,0.25698,-0.016604,-0.051752,-0.047637,-0.020670,0.001324,14.582821
3,1523948400,2018-04-17 07:00:00,ADA/USD,0.25662,0.26300,0.25489,0.25698,1215621,0.25631,-0.021549,-0.003644,0.027403,0.001324,-0.002611,14.010766
4,1523952000,2018-04-17 08:00:00,ADA/USD,0.25636,0.25998,0.25229,0.25631,896096,0.25536,-0.001014,-0.011549,-0.010253,-0.002611,-0.003713,13.705804


## add year, eth, btc


In [14]:
# Normalize year to [0, 1] scale for every 10 years
# This allows the model to extrapolate to future years naturally
df['year'] = (df['date'].dt.year - 2010) / 10

In [15]:
df = df.merge(btc, on='date', how='inner')
df = df.merge(eth, on='date', how='inner')

In [16]:
df['eth_btc_ratio'] = df['eth_close'] / df['btc_close']

In [17]:
df['btc_close_log'] = df.groupby('symbol')['btc_close'].transform(log_returns)
df['eth_close_log'] = df.groupby('symbol')['eth_close'].transform(log_returns)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 736103 entries, 0 to 736102
Data columns (total 21 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   unix            736103 non-null  int64         
 1   date            736103 non-null  datetime64[ns]
 2   symbol          736103 non-null  object        
 3   open            736103 non-null  float64       
 4   high            736103 non-null  float64       
 5   low             736103 non-null  float64       
 6   close           736103 non-null  float64       
 7   volume_usd      736103 non-null  int64         
 8   next_close      736084 non-null  float64       
 9   open_log        736084 non-null  float64       
 10  high_log        736084 non-null  float64       
 11  low_log         736084 non-null  float64       
 12  close_log       736084 non-null  float64       
 13  next_close_log  736065 non-null  float64       
 14  volume_log      736103 non-null  flo

In [18]:
df.head()

,unix,date,symbol,open,high,low,close,volume_usd,next_close,open_log,...,low_log,close_log,next_close_log,volume_log,year,btc_close,eth_close,eth_btc_ratio,btc_close_log,eth_close_log
0,1523937600,2018-04-17 04:00:00,ADA/USD,0.25551,0.28800,0.25551,0.26664,2165077,0.26200,NaN,...,NaN,NaN,NaN,14.587967,0.8,8030.00,507.60,0.063213,NaN,NaN
1,1523941200,2018-04-17 05:00:00,ADA/USD,0.26660,0.27798,0.26010,0.26200,2235633,0.25664,0.042488,...,0.017805,-0.017555,-0.020670,14.620035,0.8,7983.00,505.56,0.063330,-0.005870,-0.004027
2,1523944800,2018-04-17 06:00:00,ADA/USD,0.26221,0.26396,0.24800,0.25664,2153964,0.25698,-0.016604,...,-0.047637,-0.020670,0.001324,14.582821,0.8,8025.01,507.77,0.063273,0.005249,0.004362
3,1523948400,2018-04-17 07:00:00,ADA/USD,0.25662,0.26300,0.25489,0.25698,1215621,0.25631,-0.021549,...,0.027403,0.001324,-0.002611,14.010766,0.8,8138.45,514.28,0.063191,0.014037,0.012739
4,1523952000,2018-04-17 08:00:00,ADA/USD,0.25636,0.25998,0.25229,0.25631,896096,0.25536,-0.001014,...,-0.010253,-0.002611,-0.003713,13.705804,0.8,8131.60,516.42,0.063508,-0.000842,0.004153


## drop null-containing rows


In [19]:
# Drop all null-containing rows
df.dropna(inplace=True, ignore_index=True)

## time_idx


In [20]:
# check to make sure everything is non-null
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 736065 entries, 0 to 736064
Data columns (total 21 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   unix            736065 non-null  int64         
 1   date            736065 non-null  datetime64[ns]
 2   symbol          736065 non-null  object        
 3   open            736065 non-null  float64       
 4   high            736065 non-null  float64       
 5   low             736065 non-null  float64       
 6   close           736065 non-null  float64       
 7   volume_usd      736065 non-null  int64         
 8   next_close      736065 non-null  float64       
 9   open_log        736065 non-null  float64       
 10  high_log        736065 non-null  float64       
 11  low_log         736065 non-null  float64       
 12  close_log       736065 non-null  float64       
 13  next_close_log  736065 non-null  float64       
 14  volume_log      736065 non-null  flo

In [21]:
df['time_idx'] = df.groupby('symbol').cumcount()

# Cyclical features


In [22]:
PI = np.pi
df.sample(10)

,unix,date,symbol,open,high,low,close,volume_usd,next_close,open_log,...,close_log,next_close_log,volume_log,year,btc_close,eth_close,eth_btc_ratio,btc_close_log,eth_close_log,time_idx
191432,1525125600,2018-04-30 22:00:00,BNB/USD,14.310700,14.409000,14.065500,14.121800,1002199,14.329100,-0.009694,...,-0.013288,0.014573,13.817708,0.8,9186.02,664.50,0.072338,-0.003518,-0.006839,4142
102003,1613930400,2021-02-21 18:00:00,ATOM/USD,21.124000,21.327000,20.910000,21.294000,4167105,21.509000,0.000142,...,0.008158,0.010046,15.242732,1.1,58183.69,1964.02,0.033756,0.011837,0.007169,15909
273888,1586761200,2020-04-13 07:00:00,DENT/USD,0.000112,0.000112,0.000111,0.000111,1779,0.000111,-0.008881,...,-0.013459,0.005405,7.484369,1.0,6699.34,152.43,0.022753,-0.005395,-0.006734,5511
615104,1676826000,2023-02-19 17:00:00,RVN/USD,0.032140,0.032340,0.031490,0.031830,508013,0.031940,-0.027012,...,-0.010003,0.003450,13.138264,1.3,24361.74,1682.56,0.069066,-0.016927,-0.009617,29802
11563,1565726400,2019-08-13 20:00:00,ADA/USD,0.052410,0.052470,0.051930,0.052150,153997,0.051550,0.016933,...,-0.004401,-0.011572,11.944695,0.9,10914.24,208.02,0.019060,-0.002008,-0.001249,11563
100229,1607526000,2020-12-09 15:00:00,ATOM/USD,4.843000,4.843000,4.792000,4.812000,165163,4.822000,-0.001857,...,-0.006422,0.002076,12.014694,1.0,18349.50,567.73,0.030940,-0.005517,-0.006828,14135
18566,1591012800,2020-06-01 12:00:00,ADA/USD,0.077140,0.079270,0.076000,0.078670,4049779,0.078440,-0.011728,...,0.019899,-0.002928,15.214173,1.0,9555.01,238.90,0.025003,0.004383,0.011959,18566
578372,1672801200,2023-01-04 03:00:00,MKR/USD,512.000000,518.000000,512.000000,518.000000,245090,517.000000,0.003914,...,0.011651,-0.001932,12.409385,1.3,16862.02,1250.74,0.074175,0.007471,0.015421,21450
480399,1563541200,2019-07-19 13:00:00,LTC/USD,97.500000,97.670000,94.730000,96.400000,2719355,96.460000,0.012592,...,-0.011038,0.000622,14.815906,0.9,10303.00,215.98,0.020963,-0.016529,-0.014342,14001
606784,1646874000,2022-03-10 01:00:00,RVN/USD,0.054810,0.055370,0.053390,0.053500,323371,0.053040,-0.005277,...,-0.024556,-0.008635,12.686559,1.2,40930.71,2664.27,0.065092,-0.021278,-0.016310,21482


## hour


In [23]:
hour = lambda h: (2 * PI * h) / 24  # noqa: E731

df['hour_sin'] = np.sin(hour(df['date'].dt.hour))
df['hour_cos'] = np.cos(hour(df['date'].dt.hour))

## weekday


In [24]:
weekday = lambda w: (2 * PI * w) / 7  # noqa: E731

df['weekday_sin'] = np.sin(weekday(df['date'].dt.dayofweek))
df['weekday_cos'] = np.cos(weekday(df['date'].dt.dayofweek))

## month


In [25]:
month = lambda m: (2 * PI * m) / 12  # noqa: E731

df['month_sin'] = np.sin(month(df['date'].dt.month))
df['month_cos'] = np.cos(month(df['date'].dt.month))

## normalized day


In [26]:
norm_day = lambda nd: (2 * PI * nd)  # noqa: E731

df['norm_day_sin'] = np.sin(norm_day(df['date'].dt.day / df['date'].dt.daysinmonth))
df['norm_day_cos'] = np.cos(norm_day(df['date'].dt.day / df['date'].dt.daysinmonth))

## drop cols used for feature extraction


In [27]:
# drop columns used for feature extraction
to_drop = [
  'open',
  'high',
  'low',
  'close',
  'volume_usd',
  'next_close',
  'btc_close',
  'eth_close',
  'date',
  'unix',
]
df = df.drop(columns=to_drop)
df.columns

Index(['symbol', 'open_log', 'high_log', 'low_log', 'close_log',
       'next_close_log', 'volume_log', 'year', 'eth_btc_ratio',
       'btc_close_log', 'eth_close_log', 'time_idx', 'hour_sin', 'hour_cos',
       'weekday_sin', 'weekday_cos', 'month_sin', 'month_cos', 'norm_day_sin',
       'norm_day_cos'],
      dtype='object')

In [28]:
col_order = [
  'time_idx',
  'symbol',
  'open_log',
  'high_log',
  'low_log',
  'close_log',
  'volume_log',
  'btc_close_log',
  'eth_close_log',
  'eth_btc_ratio',
  'hour_sin',
  'hour_cos',
  'norm_day_sin',
  'norm_day_cos',
  'weekday_sin',
  'weekday_cos',
  'month_sin',
  'month_cos',
  'year',
  'next_close_log',
]
df = df.reindex(columns=col_order)

In [29]:
df.head()

,time_idx,symbol,open_log,high_log,low_log,close_log,volume_log,btc_close_log,eth_close_log,eth_btc_ratio,hour_sin,hour_cos,norm_day_sin,norm_day_cos,weekday_sin,weekday_cos,month_sin,month_cos,year,next_close_log
0,0,ADA/USD,0.042488,-0.035411,0.017805,-0.017555,14.620035,-0.005870,-0.004027,0.063330,0.965926,2.588190e-01,-0.406737,-0.913545,0.781831,0.62349,0.866025,-0.5,0.8,-0.020670
1,1,ADA/USD,-0.016604,-0.051752,-0.047637,-0.020670,14.582821,0.005249,0.004362,0.063273,1.000000,6.123234e-17,-0.406737,-0.913545,0.781831,0.62349,0.866025,-0.5,0.8,0.001324
2,2,ADA/USD,-0.021549,-0.003644,0.027403,0.001324,14.010766,0.014037,0.012739,0.063191,0.965926,-2.588190e-01,-0.406737,-0.913545,0.781831,0.62349,0.866025,-0.5,0.8,-0.002611
3,3,ADA/USD,-0.001014,-0.011549,-0.010253,-0.002611,13.705804,-0.000842,0.004153,0.063508,0.866025,-5.000000e-01,-0.406737,-0.913545,0.781831,0.62349,0.866025,-0.5,0.8,-0.003713
4,4,ADA/USD,-0.000195,-0.000693,0.002810,-0.003713,13.441326,0.001574,0.003151,0.063608,0.707107,-7.071068e-01,-0.406737,-0.913545,0.781831,0.62349,0.866025,-0.5,0.8,0.003986


In [30]:
df.shape

(736065, 20)

## Save after preprocessing done

(This takes awhile so it's commented out)


In [31]:
# df.to_csv('../data/preproc_coins.csv', index= False)